# Evaluate Random Forest with Balancing Strategies

This notebook evaluates the **Random Forest** classifier under each balancing strategy, using a reusable pipeline.

**Context from previous spot-checking notebooks:**
- `Spot_Checking_Balancing.ipynb` -> best balancing strategy (average across models): **No Balancing**
- `Spot_Checking_DimensionalityReduction.ipynb` -> best DR technique (average across models): **PCA - 90% variance**
- `Spot_Checking_ModelComparison.ipynb` -> best model under both configurations: **Random Forest**

**Goal:** Confirm whether the global best balancing strategy also holds for Random Forest in particular.

**Data flow:**
- Input: `kidney_disease_encoded.csv` -> dataset already encoded (`Encode_Categorical_Values.ipynb`)
- `MinMaxScaler` fitted only on training data of each fold (no data leakage)
- Balancing (when applicable) applied only to training data of each fold

**Approach:**
- Stratified cross-validation (`StratifiedKFold`, k=5)
- Metrics: F1, Accuracy, Precision, Recall
- Visualizations: boxplots, confusion matrices, ROC curves, ranking

**Balancing strategies evaluated:**
- **No Balancing:** training without any class adjustment
- **SMOTE:** Synthetic Minority Over-sampling Technique
- **RandomUnderSampler:** random removal of majority class samples
- **SMOTE + TomekLinks:** oversampling followed by Tomek Links cleaning


## 1. Imports

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc,
)
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

FIGURES_DIR = PROJECT_ROOT / "reports" / "paper" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

from src.modeling_utils import make_modeling_pipeline, make_stratified_cv
from src.visualization_utils import save_dataframe_as_figure

import warnings
warnings.filterwarnings('ignore')

## 2. Load and Prepare Data

In [ ]:
df = pd.read_csv("../data/processed/kidney_disease_encoded.csv")

print(f"Shape: {df.shape}")
print(f"\nTarget variable distribution:")
print(df["class"].value_counts())

In [ ]:
X = df.drop(columns=["class"])
y = LabelEncoder().fit_transform(df["class"])  # ckd=0, notckd=1 (or vice versa)

target_names = df["class"].unique().tolist()  # original labels preserved for display

print(f"Features: {X.shape[1]} columns, {X.shape[0]} samples")
print(f"Encoded classes: {np.unique(y)} (original: {df['class'].unique()})")

## 3. Pipeline

`make_pipeline(classifier, sampler=None)` is a thin wrapper around `make_modeling_pipeline` (`src.modeling_utils`) to standardize the modeling flow across balancing strategies.

**Pipeline order:**
1. **`MinMaxScaler`:** fitted only on each fold’s training split (prevents data leakage).
2. **`sampler`:** *(optional)* applied only to training data (`SMOTE`, `RandomUnderSampler`, or `SMOTETomek`); `None` means **No Balancing**.
3. **`classifier`:** the evaluated model (`RandomForestClassifier(random_state=42)` in this notebook).

In [ ]:
def make_pipeline(classifier, sampler=None):
    return make_modeling_pipeline(
        classifier=classifier,
        sampler=sampler,
    )

## 4. Balancing Strategies

In [ ]:
strategies = {
    "No Balancing":       None,
    "SMOTE":              SMOTE(random_state=42),
    "RandomUnderSampler": RandomUnderSampler(random_state=42),
    "SMOTE + TomekLinks": SMOTETomek(random_state=42),
}

## 5. Cross-Validation — Performance Metrics

In [ ]:
cv      = make_stratified_cv()
scoring = ['accuracy', 'precision', 'recall', 'f1']

# all_scores[strategy_name] = cross_validate scores dict
all_scores = {}

for strategy_name, sampler in strategies.items():
    pipeline = make_pipeline(
        classifier=RandomForestClassifier(random_state=42),
        sampler=sampler,
    )
    scores = cross_validate(pipeline, X, y, cv=cv, scoring=scoring, return_train_score=False)
    all_scores[strategy_name] = scores
    print(f"Done: {strategy_name}")

print("\nAll done.")

## 6. Summary Table

In [ ]:
rows = []
for strategy_name, scores in all_scores.items():
    row = {"Strategy": strategy_name}
    for metric in scoring:
        row[f"{metric}_mean"] = np.mean(scores[f"test_{metric}"])
        row[f"{metric}_std"]  = np.std(scores[f"test_{metric}"])
    rows.append(row)

summary_df = (
    pd.DataFrame(rows)
    .sort_values("f1_mean", ascending=False)
    .set_index("Strategy")
)

float_cols = summary_df.select_dtypes(include="float").columns.tolist()
mean_cols  = [c for c in float_cols if c.endswith("_mean")]

display(
    summary_df.style
    .format("{:.4f}", subset=float_cols)
    .background_gradient(cmap="RdYlGn", axis=0, subset=mean_cols)
)

## 7. F1-Score Boxplot (CV Folds)

In [ ]:
f1_data = {
    strategy_name: scores["test_f1"].tolist()
    for strategy_name, scores in all_scores.items()
}

fig, ax = plt.subplots(figsize=(10, 5))
pd.DataFrame(f1_data).boxplot(ax=ax)
ax.set_title("Random Forest - F1-score Distribution per Balancing Strategy (k=5 CV)")
ax.set_ylabel("F1-score")
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=15)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "evaluate_rf_balancing_boxplot.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Confusion Matrices (Aggregated over CV Folds)

Out-of-fold predictions via `cross_val_predict`: aggregates predictions from all 5 folds, giving one confusion matrix per strategy over the full dataset.

In [ ]:
n_strategies = len(strategies)
ncols = 2
nrows = int(np.ceil(n_strategies / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
axes = axes.flatten()

for ax, (strategy_name, sampler) in zip(axes, strategies.items()):
    pipeline = make_pipeline(
        classifier=RandomForestClassifier(random_state=42),
        sampler=sampler,
    )
    y_pred = cross_val_predict(pipeline, X, y, cv=cv)
    cm     = confusion_matrix(y, y_pred)
    disp   = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(strategy_name)

for ax in axes[n_strategies:]:
    ax.set_visible(False)

plt.suptitle("Random Forest - Confusion Matrices per Balancing Strategy", fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "evaluate_rf_balancing_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. ROC Curves

Out-of-fold probability predictions via `cross_val_predict(method='predict_proba')`. AUC is computed over the aggregated out-of-fold predictions (one curve per strategy).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for strategy_name, sampler in strategies.items():
    pipeline    = make_pipeline(
        classifier=RandomForestClassifier(random_state=42),
        sampler=sampler,
    )
    y_proba     = cross_val_predict(pipeline, X, y, cv=cv, method="predict_proba")
    fpr, tpr, _ = roc_curve(y, y_proba[:, 1])
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{strategy_name} (AUC = {roc_auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Random Forest - ROC Curves per Balancing Strategy")
ax.legend(loc="lower right")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "evaluate_rf_balancing_roc_curve.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Ranking

In [ ]:
# Primary criterion: highest mean F1
# Tie-breaking: lower std (more consistent across folds)
ranking_df = (
    summary_df[["f1_mean", "f1_std", "accuracy_mean", "precision_mean", "recall_mean"]]
    .sort_values(["f1_mean", "f1_std"], ascending=[False, True])
    .reset_index()
)

ranking_df.index      = ranking_df.index + 1
ranking_df.index.name = "Rank"

float_cols = ranking_df.select_dtypes(include="float").columns.tolist()
display(
    ranking_df.style
    .format("{:.4f}", subset=float_cols)
    .background_gradient(cmap="RdYlGn", axis=0, subset=float_cols)
)

save_dataframe_as_figure(
    ranking_df,
    FIGURES_DIR / "evaluate_rf_balancing_ranking.png",
)